# Academic Credential Verifier

Verify student credentials against a Neo4j graph database using Gemini AI for PDF parsing.

## 1. Install Dependencies

In [1]:
!pip install google-generativeai

In [5]:
!pip install neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 7.4 MB/s eta 0:00:00


In [18]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 62.9 MB/s eta 0:00:00


## 2. Configure Gemini AI

In [ ]:
import google.generativeai as genai

API_KEY = getpass.getpass("Enter Gemini API Key: ")

genai.configure(api_key=API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")

response = model.generate_content(
    "Say hello"
)

print(response.text)

Hello!


## 3. Connect to Neo4j

In [ ]:
from neo4j import GraphDatabase

URI = input("Enter Aura URI: ")
USERNAME = input("Enter Aura Username: ")
PASSWORD = getpass.getpass("Enter Aura Password: ")


driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

print("Connected!")

Connected!


In [7]:
with driver.session() as session:

    result = session.run("""
    MATCH (s:Student)
    RETURN s.name AS name
    """)

    for record in result:
        print(record["name"])

Bhargavi
Rahul
Aditi


## 4. Define Verification Functions

In [25]:
def verify_credential_detailed(name, university, degree, year):

    query = """
    MATCH (s:Student {name:$name})
          -[:STUDIED_AT]->(u:University),
          (s)-[:HAS_DEGREE]->(d:Degree),
          (s)-[:GRADUATED_IN]->(y:Year)

    RETURN
        u.name AS university,
        d.name AS degree,
        y.value AS year
    """

    with driver.session() as session:

        result = session.run(query, name=name)
        record = result.single()

        if record is None:
            return {
                "status": "NOT VERIFIED",
                "reason": "Student not found"
            }

        failures = []

        if record["university"] != university:
            failures.append("University mismatch")

        if record["degree"] != degree:
            failures.append("Degree mismatch")

        if record["year"] != year:
            failures.append("Year mismatch")

        if len(failures) == 0:
            return {
                "status": "VERIFIED",
                "reason": "All fields matched"
            }

        return {
            "status": "NOT VERIFIED",
            "reason": ", ".join(failures)
        }

In [ ]:
def generate_report(credential):

    result = verify_credential_detailed(
        credential["name"],
        credential["university"],
        credential["degree"],
        credential["year"]
    )

    report = f"""
Academic Credential Verification Report

Student: {credential['name']}
University: {credential['university']}
Degree: {credential['degree']}
Year: {credential['year']}

Status: {result['status']}
Reason: {result['reason']}
"""

    return report

## 6. Upload and Parse Certificate PDF

In [17]:
from google.colab import files

uploaded = files.upload()

Saving Certificate1_Version control with git and github.pdf to Certificate1_Version control with git and github.pdf


In [19]:
import fitz

pdf_file = list(uploaded.keys())[0]

doc = fitz.open(pdf_file)

text = ""

for page in doc:
    text += page.get_text()

print(text[:3000])

Mar 18 ,  2026
BHARGAVI MISRA
Version Control with Git and GitHub
an online course authorized by Board InÓnity and offered through Coursera
has successfully completed
Abhay Gupta 
Co-Founder 
Board InÓnity
Verify at: 
https://coursera.org/verify/CXX4JB5RKBCT 
  Cou rsera h as con firmed  th e id en tity  of th is in d iv id u al an d  th eir
p articip ation  in  th e cou rse.
This certiÓcate attests to the learner’s completion of an online course / project delivered via Coursera. It does not constitute formal enrollment at any university or entity and does not itself grant academic credit, grades, or
a degree. Institutions or organizations may, at their discretion, recognize this learning toward their own programs or credentials.



## 7. Extract Credentials with Gemini AI

In [14]:
prompt = f"""
Extract the following information from the certificate.

Return ONLY valid JSON.

Fields:
name
university
degree
year

Certificate:
{certificate_text}
"""

response = model.generate_content(prompt)

print(response.text)

```json
{
  "name": "Bhargavi",
  "university": "Manipal University Jaipur",
  "degree": "B.Tech CSE",
  "year": 2028
}
```


In [15]:
import json

text = response.text

text = text.replace("```json", "")
text = text.replace("```", "")

credential = json.loads(text)

print(credential)

{'name': 'Bhargavi', 'university': 'Manipal University Jaipur', 'degree': 'B.Tech CSE', 'year': 2028}


## 8. Verify Extracted Credentials

In [ ]:
result = verify_credential_detailed(
    credential["name"],
    credential["university"],
    credential["degree"],
    credential["year"]
)

print("Verification Result:", result)

Verification Result: VERIFIED


In [27]:
verification_result = verify_credential_detailed(
    credential["name"],
    credential["university"],
    credential["degree"],
    credential["year"]
)

## 9. Generate Verification Report

In [24]:
print(generate_report(credential))


Academic Credential Verification Report

Student: BHARGAVI MISRA
University: None
Degree: None
Year: 2026

Status: Student Not Found



## 10. Explanation

In [28]:
prompt = f"""
Explain the verification result.

Credential:
{credential}

Result:
{verification_result}
"""

response = model.generate_content(prompt)

print(response.text)

This verification result indicates that the system was unable to confirm the academic status of "BHARGAVI MISRA."

Here's a breakdown:

*   **`status`: 'NOT VERIFIED'**
    *   This is the overall outcome: the credential could not be authenticated or confirmed.

*   **`reason`: 'Student not found'**
    *   This is the specific explanation for the 'NOT VERIFIED' status. The system searched for a student matching the provided details but couldn't locate any record.

**Why the student might not have been found:**

The most significant factor here is the **missing `university` information** in the credential:

```
Credential:
{'name': 'BHARGAVI MISRA', 'university': None, 'degree': None, 'year': 2026}
```

1.  **No University Specified:** The verification system needs to know *which university's records to check*. Since `university` is `None`, the system has no institution to query for "BHARGAVI MISRA."
2.  **Incomplete Information:** While a name and an expected graduation year (2026) ar